In [1]:


import os
import glob
import math
import time
import random
import warnings
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

# DICOM processing
try:
    import pydicom
except ImportError:
    print("Warning: pydicom not pre-installed. Fallback mode will be used if DICOM files are missing.")
    pydicom = None

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')



# 1. GLOBAL CONFIGURATION


class Config:
    # Paths (Default Kaggle paths)
    KAGGLE_INPUT_DIR = "/kaggle/input/rsna-knee-abnormality-detection"
    KAGGLE_COMPETITIONS_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
    
    # Auto-detect input path
    if os.path.exists(KAGGLE_COMPETITIONS_DIR):
        INPUT_DIR = KAGGLE_COMPETITIONS_DIR
    else:
        INPUT_DIR = KAGGLE_INPUT_DIR

    TRAIN_CSV = os.path.join(INPUT_DIR, "train.csv")
    TEST_CSV = os.path.join(INPUT_DIR, "test.csv")
    TRAIN_SERIES_CSV = os.path.join(INPUT_DIR, "train_series.csv")
    TEST_SERIES_CSV = os.path.join(INPUT_DIR, "test_series.csv")
    TRAIN_SERIES_DIR = os.path.join(INPUT_DIR, "train_series")
    TEST_SERIES_DIR = os.path.join(INPUT_DIR, "test_series")
    
    OUTPUT_DIR = "./output"
    SUBMISSION_PATH = "submission.csv"

    # 12 Clinically Significant Knee Abnormality Targets
    TARGET_COLS = [
        "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
        "Medial OA", "Lateral OA", "PF OA", "Effusion",
        "Synovitis", "Baker's", "Contusion", "Fracture"
    ]
    NUM_CLASSES = len(TARGET_COLS)

    # Model & Data Parameters (Optimized for Fast GPU Processing)
    IMAGE_SIZE = (224, 224)
    NUM_SLICES_PER_SERIES = 4  # Key slices uniformly sampled per series (fast & accurate)
    MAX_SERIES_PER_STUDY = 2   # Top series per study (Sagittal + Coronal/Axial)
    BACKBONE_NAME = "efficientnet_b0"  # 'resnet34' or 'efficientnet_b0'
    PRETRAINED = True
    
    # Training Parameters
    SEED = 42
    N_FOLDS = 5
    EPOCHS = 4
    BATCH_SIZE = 8
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 4
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    MIXED_PRECISION = True

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True



# 2. DICOM LOADING & 2.5D PROCESSING


def load_dicom_slice(path: str, target_size: Tuple[int, int] = (224, 224)) -> np.ndarray:
    """
    Reads a DICOM file, performs rescale intercept/slope adjustment,
    contrast stretching (1st to 99th percentile), and resizes to target_size.
    Returns 2D uint8 numpy array (H, W).
    """
    if pydicom is not None and os.path.exists(path):
        try:
            dcm = pydicom.dcmread(path, stop_before_pixels=False)
            img = dcm.pixel_array.astype(np.float32)
            
            # Rescale Slope & Intercept
            slope = float(getattr(dcm, 'RescaleSlope', 1.0))
            intercept = float(getattr(dcm, 'RescaleIntercept', 0.0))
            img = img * slope + intercept
            
            # Contrast windowing (1st - 99th percentile)
            p1, p99 = np.percentile(img, (1, 99))
            if p99 > p1:
                img = np.clip(img, p1, p99)
                img = (img - p1) / (p99 - p1) * 255.0
            else:
                img = np.zeros_like(img)
            
            img_uint8 = img.astype(np.uint8)
        except Exception:
            img_uint8 = np.zeros(target_size, dtype=np.uint8)
    else:
        img_uint8 = np.zeros(target_size, dtype=np.uint8)
    
    # Bilinear resize to target_size
    tensor_img = torch.from_numpy(img_uint8).unsqueeze(0).unsqueeze(0).float()
    tensor_resized = F.interpolate(tensor_img, size=target_size, mode='bilinear', align_corners=False)
    return tensor_resized.squeeze().numpy().astype(np.uint8)


def build_25d_triplets(slice_paths: List[str], n_samples: int, target_size: Tuple[int, int]) -> torch.Tensor:
    """
    Uniformly samples n_samples slice indices and creates 3-channel 2.5D triplets [i-1, i, i+1].
    Normalizes with standard ImageNet statistics.
    Returns Tensor of shape (n_samples, 3, H, W).
    """
    num_slices = len(slice_paths)
    if num_slices == 0:
        return torch.zeros((n_samples, 3, target_size[0], target_size[1]), dtype=torch.float32)

    # Uniform keyframe sampling across volume
    if num_slices >= n_samples:
        indices = np.linspace(0, num_slices - 1, n_samples, dtype=int)
    else:
        indices = np.pad(np.arange(num_slices), (0, n_samples - num_slices), mode='edge')

    sampled_tensors = []
    mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)

    for idx in indices:
        prev_idx = max(0, idx - 1)
        curr_idx = idx
        next_idx = min(num_slices - 1, idx + 1)

        img_prev = load_dicom_slice(slice_paths[prev_idx], target_size)
        img_curr = load_dicom_slice(slice_paths[curr_idx], target_size)
        img_next = load_dicom_slice(slice_paths[next_idx], target_size)

        triplet = np.stack([img_prev, img_curr, img_next], axis=0).astype(np.float32) / 255.0
        triplet = (triplet - mean) / std

        sampled_tensors.append(torch.from_numpy(triplet).float())

    return torch.stack(sampled_tensors, dim=0)  # (N, 3, H, W)



# 3. PYTORCH DATASET


class RSNAKneeDataset(Dataset):
    """
    Multi-Series Dataset for RSNA Knee Abnormality Detection.
    Loads study data, identifies series, samples 2.5D slices per series.
    """
    def __init__(self, df: pd.DataFrame, series_df: Optional[pd.DataFrame], base_dir: str, config: Config, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.base_dir = base_dir
        self.config = config
        self.is_train = is_train

        # Map StudyInstanceUID -> List of SeriesInstanceUIDs
        self.study_to_series = {}
        if series_df is not None and not series_df.empty and 'StudyInstanceUID' in series_df.columns:
            grouped = series_df.groupby('StudyInstanceUID')
            for study_id, group in grouped:
                self.study_to_series[study_id] = group['SeriesInstanceUID'].tolist()
        
        self.series_cache = {}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        study_id = row['StudyInstanceUID']

        # Get target labels (replace any stray NaNs with 0.0)
        labels = torch.zeros(self.config.NUM_CLASSES, dtype=torch.float32)
        if self.is_train and all(c in row for c in self.config.TARGET_COLS):
            val_array = row[self.config.TARGET_COLS].fillna(0.0).values.astype(np.float32)
            labels = torch.tensor(val_array)

        # Retrieve series for this study
        series_ids = self.study_to_series.get(study_id, [])
        study_path = os.path.join(self.base_dir, study_id)
        if not series_ids and os.path.exists(study_path):
            series_ids = [d for d in os.listdir(study_path) if os.path.isdir(os.path.join(study_path, d))]
        
        series_ids = series_ids[:self.config.MAX_SERIES_PER_STUDY]
        
        study_tensors = []
        for s_id in series_ids:
            series_dir = os.path.join(self.base_dir, study_id, s_id)
            if series_dir in self.series_cache:
                dcm_files = self.series_cache[series_dir]
            elif os.path.exists(series_dir):
                dcm_files = sorted([os.path.join(series_dir, f) for f in os.listdir(series_dir) if f.endswith(".dcm")])
                self.series_cache[series_dir] = dcm_files
            else:
                dcm_files = []
            
            series_tensor = build_25d_triplets(dcm_files, self.config.NUM_SLICES_PER_SERIES, self.config.IMAGE_SIZE)
            study_tensors.append(series_tensor)

        # Pad missing series up to MAX_SERIES_PER_STUDY
        while len(study_tensors) < self.config.MAX_SERIES_PER_STUDY:
            empty_series = torch.zeros(
                (self.config.NUM_SLICES_PER_SERIES, 3, self.config.IMAGE_SIZE[0], self.config.IMAGE_SIZE[1]),
                dtype=torch.float32
            )
            study_tensors.append(empty_series)

        # Output shape: (S, N, 3, H, W)
        study_tensor = torch.stack(study_tensors, dim=0)
        return study_tensor, labels, study_id



# 4. NEURAL NETWORK ARCHITECTURE


class GatedAttentionPooling(nn.Module):
    """
    Gated Attention mechanism to pool slice/series feature representations.
    Learns to dynamically weight slices exhibiting pathological abnormalities.
    """
    def __init__(self, in_features: int, hidden_dim: int = 128):
        super().__init__()
        self.attn_w = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        self.attn_v = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.Sigmoid(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # x: (B, M, Feature_Dim) where M = S * N
        w = self.attn_w(x)
        v = self.attn_v(x)
        scores = w * v
        weights = F.softmax(scores, dim=1)
        pooled = torch.sum(x * weights, dim=1)
        return pooled, weights


class RSNAKneeModel(nn.Module):
    """
    2.5D Multimodal Architecture:
      1. 2D CNN Feature Extractor (EfficientNet-B0 / ResNet-34)
      2. Gated Attention Sequence Aggregation across all series/slices
      3. 12-Target Multi-Label Classification Head
    """
    def __init__(self, backbone_name: str = "efficientnet_b0", num_classes: int = 12, pretrained: bool = True):
        super().__init__()
        self.num_classes = num_classes

        # Backbone Selection
        if "resnet" in backbone_name:
            try:
                weights = models.ResNet34_Weights.DEFAULT if pretrained else None
                resnet = models.resnet34(weights=weights)
            except AttributeError:
                resnet = models.resnet34(pretrained=pretrained)
            self.feature_dim = resnet.fc.in_features
            resnet.fc = nn.Identity()
            self.backbone = resnet
        else:
            try:
                weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
                effnet = models.efficientnet_b0(weights=weights)
            except AttributeError:
                effnet = models.efficientnet_b0(pretrained=pretrained)
            self.feature_dim = effnet.classifier[1].in_features
            effnet.classifier = nn.Identity()
            self.backbone = effnet

        self.dropout = nn.Dropout(0.3)
        self.attention = GatedAttentionPooling(in_features=self.feature_dim, hidden_dim=256)
        
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, S, N, 3, H, W)
        B, S, N, C, H, W = x.shape
        x_flat = x.view(B * S * N, C, H, W)
        
        features = self.backbone(x_flat)
        features = self.dropout(features)
        
        features_seq = features.view(B, S * N, self.feature_dim)
        pooled_features, _ = self.attention(features_seq)
        
        logits = self.classifier(pooled_features)
        return logits



# 5. METRIC EVALUATION (NaN-SAFE MACRO AUC)


def calculate_macro_auc(y_true: np.ndarray, y_pred: np.ndarray, target_cols: List[str]) -> Tuple[float, Dict[str, float]]:
    """
    Computes Macro AUC ROC score across the 12 target columns.
    Defensively removes any NaNs and handles edge cases where only 1 class is present.
    """
    aucs = {}
    valid_aucs = []
    
    for i, col in enumerate(target_cols):
        y_t = y_true[:, i]
        y_p = y_pred[:, i]
        
        # Filter out any NaN entries
        mask = ~np.isnan(y_t)
        y_t_clean = y_t[mask]
        y_p_clean = y_p[mask]
        
        # Calculate AUC if both positive and negative instances exist
        if len(y_t_clean) > 0 and len(np.unique(y_t_clean)) > 1:
            try:
                score = roc_auc_score(y_t_clean, y_p_clean)
                aucs[col] = float(score)
                valid_aucs.append(score)
            except ValueError:
                aucs[col] = 0.5
        else:
            aucs[col] = 0.5  # Neutral fallback for single-class validation folds

    macro_auc = float(np.mean(valid_aucs)) if valid_aucs else 0.5
    return macro_auc, aucs



# 6. TRAINING & VALIDATION LOOPS


def train_one_epoch(model, loader, criterion, optimizer, scaler, scheduler, device):
    model.train()
    total_loss = 0.0
    num_batches = len(loader)
    
    for step, (images, targets, _) in enumerate(loader):
        images = images.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(images)
                loss = criterion(logits, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        
        # Live batch progress logging every 100 steps
        if (step + 1) % 100 == 0 or (step + 1) == num_batches:
            avg = total_loss / (step + 1)
            pct = (step + 1) / num_batches * 100
            print(f"   Batch [{step+1:04d}/{num_batches:04d}] ({pct:5.1f}%) - Current Loss: {avg:.4f}", flush=True)

    if scheduler is not None:
        scheduler.step()

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def validate(model, loader, criterion, device, target_cols):
    model.eval()
    total_loss = 0.0
    all_targets = []
    all_preds = []
    
    for images, targets, _ in loader:
        images = images.to(device)
        targets = targets.to(device)
        
        logits = model(images)
        loss = criterion(logits, targets)
        probs = torch.sigmoid(logits)
        
        total_loss += loss.item()
        all_targets.append(targets.cpu().numpy())
        all_preds.append(probs.cpu().numpy())

    all_targets = np.vstack(all_targets) if all_targets else np.zeros((0, len(target_cols)))
    all_preds = np.vstack(all_preds) if all_preds else np.zeros((0, len(target_cols)))
    
    macro_auc, class_aucs = calculate_macro_auc(all_targets, all_preds, target_cols)
    avg_loss = total_loss / max(len(loader), 1)
    
    return avg_loss, macro_auc, class_aucs



# 7. CROSS-VALIDATION PIPELINE


def run_cross_validation(train_df: pd.DataFrame, series_df: Optional[pd.DataFrame], config: Config):
    # Fill missing target values with 0.0 (absence of abnormality) to utilize the full dataset
    train_df[config.TARGET_COLS] = train_df[config.TARGET_COLS].fillna(0.0)
    labeled_df = train_df.reset_index(drop=True)
    
    print("=" * 60)
    print("RSNA Knee Abnormality Detection: Cross-Validation")
    print(f"Total raw studies: {len(train_df)} | Active Training studies: {len(labeled_df)}")
    print(f"Running {config.N_FOLDS}-Fold Training")
    print("=" * 60)

    if len(labeled_df) == 0:
        raise ValueError("No labeled studies found! Check config.TARGET_COLS or your train.csv.")

    kf = KFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)
    
    oof_predictions = np.zeros((len(labeled_df), config.NUM_CLASSES))
    oof_targets = labeled_df[config.TARGET_COLS].values
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(labeled_df)):
        print(f"\n--- Fold {fold + 1}/{config.N_FOLDS} (Train: {len(train_idx)}, Val: {len(val_idx)}) ---")
        
        fold_train_df = labeled_df.iloc[train_idx].reset_index(drop=True)
        fold_val_df = labeled_df.iloc[val_idx].reset_index(drop=True)
        
        train_ds = RSNAKneeDataset(fold_train_df, series_df, config.TRAIN_SERIES_DIR, config, is_train=True)
        val_ds = RSNAKneeDataset(fold_val_df, series_df, config.TRAIN_SERIES_DIR, config, is_train=True)
        
        train_loader = DataLoader(
            train_ds, batch_size=config.BATCH_SIZE, shuffle=True, 
            num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=(len(train_ds) > config.BATCH_SIZE)
        )
        val_loader = DataLoader(
            val_ds, batch_size=config.BATCH_SIZE, shuffle=False, 
            num_workers=config.NUM_WORKERS, pin_memory=True
        )
        
        # Model, Loss, Optimizer
        model = RSNAKneeModel(backbone_name=config.BACKBONE_NAME, num_classes=config.NUM_CLASSES, pretrained=config.PRETRAINED)
        model = model.to(config.DEVICE)
        
        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.EPOCHS, eta_min=1e-6)
        scaler = torch.cuda.amp.GradScaler() if (config.MIXED_PRECISION and config.DEVICE == "cuda") else None
        
        best_val_auc = 0.0
        os.makedirs(config.OUTPUT_DIR, exist_ok=True)
        model_save_path = os.path.join(config.OUTPUT_DIR, f"model_fold_{fold}.pth")

        for epoch in range(config.EPOCHS):
            t0 = time.time()
            train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, scheduler, config.DEVICE)
            val_loss, val_auc, _ = validate(model, val_loader, criterion, config.DEVICE, config.TARGET_COLS)
            elapsed = time.time() - t0
            
            print(f"Epoch {epoch+1:02d}/{config.EPOCHS:02d} [{elapsed:.1f}s] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro AUC: {val_auc:.4f}", flush=True)
            
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                torch.save(model.state_dict(), model_save_path)
                print(f"  -> Saved Checkpoint (Best AUC: {best_val_auc:.4f})", flush=True)

        # Load best fold model for OOF evaluation
        if os.path.exists(model_save_path):
            model.load_state_dict(torch.load(model_save_path, map_location=config.DEVICE))
        model.eval()
        
        fold_preds = []
        with torch.no_grad():
            for images, _, _ in val_loader:
                images = images.to(config.DEVICE)
                logits = model(images)
                probs = torch.sigmoid(logits)
                fold_preds.append(probs.cpu().numpy())
                
        if fold_preds:
            oof_predictions[val_idx] = np.vstack(fold_preds)

    total_oof_auc, class_aucs = calculate_macro_auc(oof_targets, oof_predictions, config.TARGET_COLS)
    print("\n" + "=" * 60)
    print(f"FINAL OVERALL OOF MACRO AUC ROC: {total_oof_auc:.4f}")
    print("=" * 60)
    for col, score in class_aucs.items():
        print(f"  - {col:18s}: {score:.4f}")
    print("=" * 60)



# 8. INFERENCE & KAGGLE SUBMISSION ENGINE


def generate_submission(config: Config):
    """
    Generates submission.csv for Kaggle hidden test set.
    Ensembles all available trained fold checkpoints.
    """
    print("\n" + "=" * 60)
    print("Generating Final Test Submission")
    print("=" * 60)

    if not os.path.exists(config.TEST_CSV):
        print(f"Test CSV path '{config.TEST_CSV}' not found. Generating sample fallback submission.")
        create_dummy_submission(config)
        return

    test_df = pd.read_csv(config.TEST_CSV)
    test_series_df = pd.read_csv(config.TEST_SERIES_CSV) if os.path.exists(config.TEST_SERIES_CSV) else None

    test_ds = RSNAKneeDataset(test_df, test_series_df, config.TEST_SERIES_DIR, config, is_train=False)
    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)

    # Search for fold checkpoints
    model_paths = sorted(glob.glob(os.path.join(config.OUTPUT_DIR, "model_fold_*.pth")))
    
    if not model_paths:
        print("No trained checkpoints found! Generating default baseline predictions (0.5).")
        for col in config.TARGET_COLS:
            test_df[col] = 0.5
        test_df[['StudyInstanceUID'] + config.TARGET_COLS].to_csv(config.SUBMISSION_PATH, index=False)
        print(f"Saved fallback to {config.SUBMISSION_PATH}")
        return

    print(f"Found {len(model_paths)} model checkpoint(s) for ensemble inference.")
    all_model_preds = np.zeros((len(test_df), config.NUM_CLASSES), dtype=np.float32)

    for p in model_paths:
        print(f"Running inference with model: {os.path.basename(p)}")
        model = RSNAKneeModel(backbone_name=config.BACKBONE_NAME, num_classes=config.NUM_CLASSES, pretrained=False)
        model.load_state_dict(torch.load(p, map_location=config.DEVICE))
        model.to(config.DEVICE)
        model.eval()

        fold_preds = []
        with torch.no_grad():
            for images, _, _ in test_loader:
                images = images.to(config.DEVICE)
                logits = model(images)
                probs = torch.sigmoid(logits)
                fold_preds.append(probs.cpu().numpy())

        fold_preds_arr = np.vstack(fold_preds)
        all_model_preds += fold_preds_arr / len(model_paths)

    sub_df = pd.DataFrame({'StudyInstanceUID': test_df['StudyInstanceUID']})
    for i, col in enumerate(config.TARGET_COLS):
        sub_df[col] = np.clip(all_model_preds[:, i], 0.0, 1.0)

    sub_df.to_csv(config.SUBMISSION_PATH, index=False)
    print(f"Successfully generated Kaggle submission: '{config.SUBMISSION_PATH}'")
    print(f"Submission Shape: {sub_df.shape}")
    print("First 3 rows:")
    print(sub_df.head(3))


def create_dummy_submission(config: Config):
    """Fallback dummy submission creator matching exact RSNA Kaggle submission schema."""
    dummy_data = {
        'StudyInstanceUID': ['test_study_001', 'test_study_002', 'test_study_003']
    }
    for col in config.TARGET_COLS:
        dummy_data[col] = [0.5, 0.5, 0.5]
    
    sub = pd.DataFrame(dummy_data)
    sub.to_csv(config.SUBMISSION_PATH, index=False)
    print(f"Created fallback dummy submission: {config.SUBMISSION_PATH}")



# 9. MAIN EXECUTION


if __name__ == "__main__":
    seed_everything(Config.SEED)
    
    if os.path.exists(Config.TRAIN_CSV):
        print("Kaggle training data detected. Loading datasets...")
        train_df = pd.read_csv(Config.TRAIN_CSV)
        series_df = pd.read_csv(Config.TRAIN_SERIES_CSV) if os.path.exists(Config.TRAIN_SERIES_CSV) else None
        
        # Run 5-fold training
        run_cross_validation(train_df, series_df, Config)
        
        # Generate submission
        generate_submission(Config)
    else:
        print("Dataset not found at Kaggle path. Generating fallback submission.")
        generate_submission(Config)


Kaggle training data detected. Loading datasets...
RSNA Knee Abnormality Detection: Cross-Validation
Total raw studies: 4407 | Active Training studies: 4407
Running 5-Fold Training

--- Fold 1/5 (Train: 3525, Val: 882) ---
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 215MB/s]


   Batch [0100/0440] ( 22.7%) - Current Loss: 0.1128
   Batch [0200/0440] ( 45.5%) - Current Loss: 0.0762
   Batch [0300/0440] ( 68.2%) - Current Loss: 0.0641
   Batch [0400/0440] ( 90.9%) - Current Loss: 0.0532
   Batch [0440/0440] (100.0%) - Current Loss: 0.0511
Epoch 01/04 [491.5s] - Train Loss: 0.0511 | Val Loss: 0.0232 | Val Macro AUC: 0.5963
  -> Saved Checkpoint (Best AUC: 0.5963)
   Batch [0100/0440] ( 22.7%) - Current Loss: 0.0631
   Batch [0200/0440] ( 45.5%) - Current Loss: 0.0482
   Batch [0300/0440] ( 68.2%) - Current Loss: 0.0401
   Batch [0400/0440] ( 90.9%) - Current Loss: 0.0359
   Batch [0440/0440] (100.0%) - Current Loss: 0.0350
Epoch 02/04 [458.8s] - Train Loss: 0.0350 | Val Loss: 0.0229 | Val Macro AUC: 0.5192
   Batch [0100/0440] ( 22.7%) - Current Loss: 0.0412
   Batch [0200/0440] ( 45.5%) - Current Loss: 0.0318
   Batch [0300/0440] ( 68.2%) - Current Loss: 0.0319
   Batch [0400/0440] ( 90.9%) - Current Loss: 0.0288
   Batch [0440/0440] (100.0%) - Current Loss: 0